# Mission : Pilotez l'atterrisseur lunaire Eagle-1


## Installation des dépendances

Ce notebook est exécuté sur Google Colab, la cellule suivante installe les dépendances nécessaires.

In [1]:
!pip install gymnasium[box2d] stable-baselines3 "imageio[ffmpeg]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 56.8 MB/s eta 0:00:00


In [2]:
import random
import numpy as np
import torch
import gymnasium as gym
import imageio
import zipfile
from PIL import Image, ImageDraw
from IPython.display import Video
from google.colab import files
from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Création de l'environnement

In [3]:
# Create environment
env = gym.make("LunarLander-v3")

print(f"Observation space: {env.observation_space}")
print(f"Observation shape: {env.observation_space.shape}")
print(f"Observation sample: {env.observation_space.sample()}")

print(f"Action space: {env.action_space}")
print(f"Action sample: {env.action_space.sample()}")
print(f"Nombre d'actions: {env.action_space.n}")

Observation space: Box([ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ], [ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ], (8,), float32)
Observation shape: (8,)
Observation sample: [ 0.29981163  1.8088123   7.182409   -2.2121866   0.67503434  3.643541
  0.20767257  0.11196628]
Action space: Discrete(4)
Action sample: 2
Nombre d'actions: 4


<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type swigvarlink has no __module__ attribute


**Observation de l'environnement**

L'espace d'observation contient 8 variables continues : position (x, y),
vitesse (x, y), angle, vitesse angulaire, et 2 capteurs de contact (booléens).

L'espace d'action est discret avec 4 actions possibles :
- 0 : ne rien faire
- 1 : allumer le propulseur gauche
- 2 : allumer le propulseur principal (bas)
- 3 : allumer le propulseur droit

## Choix de l'algorithme

L'espace d'action de LunarLander-v3 est discret (4 actions), ce qui oriente le choix vers deux algorithmes compatibles avec Stable-Baselines3.

**DQN (Deep Q-Network)** apprend une fonction Q(s, a) qui estime la récompense future pour chaque action depuis un état donné. Il sélectionne toujours l'action avec la valeur Q la plus élevée et stocke ses expériences dans un replay buffer pour stabiliser l'apprentissage (off-policy).

**PPO (Proximal Policy Optimization)** apprend directement une politique : une distribution de probabilités sur les actions à prendre. Il ajoute une contrainte de clipping pour éviter des mises à jour trop brusques (on-policy). Bien que conçu pour les espaces continus, il fonctionne aussi sur du discret.

On part sur **DQN** pour cette mission : il est explicitement recommandé par le brief pour les espaces d'action discrets, et a déjà été appliqué sur CartPole en exercice 3, le pipeline est donc connu.

In [4]:
def create_model_DQN(policy, env, **kwargs):
    model = DQN(policy, env, verbose=1, tensorboard_log="./logs/", seed=SEED, **kwargs)
    return model


def learn_and_evaluated_model(model, env, total_timesteps=500000, n_eval_episodes=50):
    model.learn(total_timesteps)
    mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes)
    print(f"Récompense moyenne : {mean_reward:.2f} +/- {std_reward:.2f}")
    return mean_reward, std_reward

In [5]:
model_baseline = create_model_DQN('MlpPolicy', env)
learn_and_evaluated_model(model_baseline, env)

model_baseline.save("dqn_baseline")
print("Modèle sauvegardé : dqn_baseline.zip")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to ./logs/DQN_1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 85       |
|    ep_rew_mean      | -174     |
|    exploration_rate | 0.994    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 839      |
|    time_elapsed     | 0        |
|    total_timesteps  | 340      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 4.59     |
|    n_updates        | 59       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 94       |
|    ep_rew_mean      | -141     |
|    exploration_rate | 0.986    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 1017     |
|    time_elapsed     | 0        |
|    total_timesteps  | 752      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 1.88     |
|    n_updates      

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Récompense moyenne : 59.43 +/- 127.45
Modèle sauvegardé : dqn_baseline.zip


**Modèle baseline**

Entraînement : 500 000 timesteps, hyperparamètres par défaut SB3.
Récompense moyenne : 59.43 +/- 127.45 sur 50 épisodes.

L'écart-type élevé (127.45), supérieur à la moyenne elle-même, traduit une politique quasi aléatoire : l'agent réussit certains atterrissages par hasard mais s'écrase ou sort des limites sur la majorité des épisodes. Le score moyen reste loin de l'objectif de 200. L'optimisation des hyperparamètres est nécessaire.

## Optimisation des hyperparamètres

La stratégie : modifier un seul hyperparamètre à la fois, évaluer sur 50 épisodes, puis passer au suivant en conservant les changements bénéfiques.

Paramètres explorés :

`learning_starts` : nombre de steps aléatoires avant que l'agent commence à apprendre. Le défaut SB3 est 50 000, soit 10 % du budget total à ne rien apprendre. On le réduit à 10 000.

`learning_rate` : vitesse à laquelle le réseau corrige ses poids après chaque erreur. Trop élevé : instabilité. Trop faible : convergence lente. Défaut SB3 : 1e-4.

`batch_size` : taille de l'échantillon tiré du buffer à chaque mise à jour. Un batch trop petit produit un signal bruité. Défaut SB3 : 32.

`target_update_interval` : fréquence de synchronisation du réseau cible. DQN maintient deux réseaux, un actif (qui apprend) et un cible (référence stable pour calculer les erreurs). Défaut SB3 : 10 000.

`buffer_size` : capacité de la mémoire d'expériences. 1 million par défaut est disproportionné pour 500k steps. On réduit à 50 000.

### Expérience 1 (learning_starts : 50 000 → 10 000)


In [6]:
model_exp1 = create_model_DQN('MlpPolicy', env, learning_starts=10000)
learn_and_evaluated_model(model_exp1, env)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to ./logs/DQN_2
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 85.5     |
|    ep_rew_mean      | -191     |
|    exploration_rate | 0.994    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 3643     |
|    time_elapsed     | 0        |
|    total_timesteps  | 342      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 90.8     |
|    ep_rew_mean      | -179     |
|    exploration_rate | 0.986    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 3759     |
|    time_elapsed     | 0        |
|    total_timesteps  | 726      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 96.3     |
|    ep

(np.float64(4.113559139500783), np.float64(98.67326728460648))

**Observation :**

Résultat : 4.11 +/- 98.67 sur 50 épisodes.

Régression par rapport au baseline (59.43). Réduire `learning_starts` de 50 000 à 10 000 donne à l'agent 40 000 steps supplémentaires d'apprentissage réel, mais avec le `learning_rate` par défaut (1e-4, très prudent) et un `target_update_interval` toujours à 10 000, ces steps supplémentaires ne servent à rien : le réseau apprend plus tôt, mais trop lentement pour en tirer parti, sur une cible encore quasi figée.

Ce paramètre seul ne suffit donc pas : il a besoin d'un `learning_rate` plus élevé pour être exploité. C'est le test suivant.

### Expérience 2 (learning_rate : 1e-4 → 6.3e-4)


In [7]:
model_exp2 = create_model_DQN(
    'MlpPolicy', env,
    learning_starts=10000,
    learning_rate=6.3e-4,
)
learn_and_evaluated_model(model_exp2, env)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Logging to ./logs/DQN_3
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 85.5     |
|    ep_rew_mean      | -191     |
|    exploration_rate | 0.994    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 4937     |
|    time_elapsed     | 0        |
|    total_timesteps  | 342      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 90.8     |
|    ep_rew_mean      | -179     |
|    exploration_rate | 0.986    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 5430     |
|    time_elapsed     | 0        |
|    total_timesteps  | 726      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 96.3     |
|    ep

(np.float64(223.6238134508584), np.float64(62.262012375964005))

**Observation :**

Résultat : 223.62 +/- 62.26 sur 50 épisodes.

Net progrès par rapport au baseline (59.43) et à l'expérience 1 (4.11). Le `learning_rate` élevé permet au réseau d'exploiter les steps d'apprentissage gagnés en réduisant `learning_starts` en expérience 1 : seul, ce paramètre était inutile, combiné à un LR plus rapide, il devient le principal levier de progression testé jusqu'ici.

L'écart-type chute nettement (98.67 → 62.26), signe d'une politique plus cohérente d'un épisode à l'autre.

Malgré ce résultat déjà satisfaisant, l'expérimentation se poursuit : le replay buffer (`batch_size`, `target_update_interval`, `buffer_size`) reste un axe du DQN non encore testé, et il reste à vérifier qu'elle tient toujours sur un entraînement plus long, celui prévu pour le modèle final.

### Expérience 3 (batch_size, target_update_interval, buffer_size)

Ces trois paramètres gouvernent la mécanique du replay buffer : la taille des lots d'apprentissage, la fréquence de synchronisation du réseau cible, et la capacité mémoire. Ils sont regroupés car ils interagissent directement : augmenter le `batch_size` sans réduire le `target_update_interval` produit des effets contradictoires.

Changements :
- `batch_size` 32 → 128
- `target_update_interval` 10 000 → 250
- `buffer_size` 1 000 000 → 50 000


In [8]:
model_exp3 = create_model_DQN(
    'MlpPolicy', env,
    learning_starts=10000,
    learning_rate=6.3e-4,
    batch_size=128,
    target_update_interval=250,
    buffer_size=50000,
)
learn_and_evaluated_model(model_exp3, env)

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
|    learning_rate    | 0.00063  |
|    loss             | 1.97     |
|    n_updates        | 1035     |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 108      |
|    ep_rew_mean      | -165     |
|    exploration_rate | 0.723    |
| time/               |          |
|    episodes         | 140      |
|    fps              | 2111     |
|    time_elapsed     | 6        |
|    total_timesteps  | 14566    |
| train/              |          |
|    learning_rate    | 0.00063  |
|    loss             | 1.81     |
|    n_updates        | 1141     |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 110      |
|    ep_rew_mean      | -161     |
|    exploration_rate | 0.713    |
| time/               |          |
|    episodes         | 144      |
|    fps    

(np.float64(163.15346745747033), np.float64(93.87874877517643))

**Observation :**

Résultat : 163.15 +/- 93.88 sur 50 épisodes.

Régression par rapport à l'expérience 2 (223.62). Des batchs plus grands (128) et un buffer plus resserré (50 000, expériences plus récentes et pertinentes) auraient dû aider, mais `target_update_interval=250` est trop agressif : resynchroniser le réseau cible toutes les 250 steps au lieu de 10 000 supprime une grande partie de son rôle stabilisateur, qui est justement de fournir une référence semi-fixe pour calculer l'erreur. Ce constat, confirmé plus loin dans ce notebook, guidera la correction apportée au modèle final.

Le score (163.15) et l'écart-type (93.88) régressent tous les deux par rapport à l'expérience 2, ce qui montre que regrouper trois changements à la fois masque leurs effets individuels contradictoires.

## Modèle final

In [9]:
model_final = create_model_DQN(
    'MlpPolicy', env,
    learning_starts=1000,
    learning_rate=6.3e-4,
    batch_size=128,
    target_update_interval=1000,
    buffer_size=50000,
)
mean_reward, std_reward = learn_and_evaluated_model(
    model_final, env,
    total_timesteps=1_000_000,
    n_eval_episodes=100,
)

model_final.save("dqn_final")
print("Modèle sauvegardé : dqn_final.zip")


Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
|    loss             | 1.82     |
|    n_updates        | 142617   |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 169      |
|    ep_rew_mean      | 83.9     |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 1860     |
|    fps              | 820      |
|    time_elapsed     | 697      |
|    total_timesteps  | 572192   |
| train/              |          |
|    learning_rate    | 0.00063  |
|    loss             | 0.363    |
|    n_updates        | 142797   |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 178      |
|    ep_rew_mean      | 91.6     |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 1864     |
|    fps              | 820      |
|    time_el

**Observation :**

Résultat : 252.04 +/- 68.30 sur 100 épisodes.

Objectif largement atteint (≥ 200), meilleur résultat de toute la série. Le modèle final reprend les ingrédients utiles de l'expérience 3 (`batch_size=128`, `buffer_size=50 000`) mais corrige `target_update_interval` à 1000 au lieu de 250 : toujours plus fréquent que le défaut SB3, sans supprimer l'effet stabilisateur du réseau cible. Combiné à `learning_starts=1000` (encore plus tôt qu'en expérience 1) et à 1 million de steps, cette configuration convertit enfin le potentiel du LR élevé (déjà visible en expérience 2) en un résultat stable.

L'écart-type (68.30) est le plus bas de toute la série, signe d'un agent qui atterrit de manière cohérente plutôt qu'aléatoire. Le modèle est sauvegardé sous `dqn_final.zip`.

## Génération de la vidéo

Un seul atterrissage réussi ne dure qu'environ 6 secondes, ce qui est trop court pour le livrable attendu (20 à 30s).

On enchaîne donc plusieurs épisodes réussis (récompense ≥ 200) jusqu'à atteindre la durée cible. À chaque frame, la récompense cumulée de l'épisode en cours est dessinée directement sur l'image avec PIL avant d'être stockée. `imageio` assemble ensuite l'ensemble des frames en fichier `.mp4`.

In [14]:
model = DQN.load("dqn_final.zip")
env = gym.make("LunarLander-v3", render_mode="rgb_array")

FPS = 30
TARGET_DURATION = 25  # secondes, dans la fourchette 20-30s demandée
TARGET_FRAMES = FPS * TARGET_DURATION

video_frames = []
successful_landings = 0

for attempt in range(30):
    if len(video_frames) >= TARGET_FRAMES:
        break

    obs, _ = env.reset()
    frames = []
    total_reward = 0.0
    terminated = False
    truncated = False

    while not terminated and not truncated:
        frame = env.render()

        # Superposition de la récompense cumulée sur la frame brute
        img = Image.fromarray(frame)
        draw = ImageDraw.Draw(img)
        draw.text((10, 10), f"Récompense : {total_reward:.1f}", fill=(255, 255, 255))
        frames.append(np.array(img))

        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward

    print(f"Essai {attempt + 1} : {total_reward:.1f}")

    if total_reward >= 200:
        successful_landings += 1
        video_frames.extend(frames)
        print(f"Atterrissage réussi #{successful_landings}, {len(video_frames)} frames cumulées ({len(video_frames) / FPS:.1f}s).")

env.close()

imageio.mimsave("eagle1_landing.mp4", video_frames, fps=FPS)
print(f"Vidéo sauvegardée : eagle1_landing.mp4 ({len(video_frames) / FPS:.1f}s, {successful_landings} atterrissages)")

Essai 1 : 305.7
Atterrissage réussi #1, 226 frames cumulées (7.5s).
Essai 2 : 289.9
Atterrissage réussi #2, 473 frames cumulées (15.8s).
Essai 3 : 256.9
Atterrissage réussi #3, 650 frames cumulées (21.7s).


Essai 4 : 304.8
Atterrissage réussi #4, 902 frames cumulées (30.1s).
Vidéo sauvegardée : eagle1_landing.mp4 (30.1s, 4 atterrissages)


In [15]:
Video("eagle1_landing.mp4", embed=True)

In [16]:
files.download("eagle1_landing.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Formatage des données

Le schéma ci-dessous montre le flux complet entre les trois composants du système.

![Flux état → action](mermaid-diagram-2026-06-26-113655.png)

**obs.tolist()** : Gymnasium retourne un `np.ndarray float32`. JSON ne sait pas sérialiser ce type directement, `.tolist()` le convertit en liste Python native.

**int(action)** : `model.predict()` retourne un `np.int64`. FastAPI ne peut pas le sérialiser en JSON tel quel, `int()` le convertit en entier Python natif.

**Récompenses vers le dashboard** : à chaque épisode, `total_reward` et `step_count` sont stockés dans une liste de dicts, puis convertis en DataFrame pandas pour alimenter les graphiques Plotly.

In [13]:
with zipfile.ZipFile("dqn_models.zip", "w") as zf:
    zf.write("dqn_baseline.zip")
    zf.write("dqn_final.zip")

files.download("dqn_models.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Déploiement local

Le modèle entraîné est exposé via trois composants distincts, à lancer en local depuis la racine du dépôt. Chaque composant a un rôle précis :

| Composant | Fichier | Rôle |
|---|---|---|
| API FastAPI | `api/main.py` | Charge le modèle et expose `/play` (état → action) |
| GUI Streamlit | `gui/app.py` | Visualise un épisode joué en temps réel |
| Dashboard Streamlit | `dashboard/app.py` | Agrège les métriques sur N épisodes |

**Prérequis** : placer `dqn_final.zip` dans `notebook/models/` (téléchargé depuis Colab via la cellule ci-dessus).

### 1. Lancer l'API

L'API doit être démarrée en premier : le GUI et le dashboard en dépendent.

```bash
uvicorn api.main:app --reload
```

L'API est accessible sur `http://localhost:8000`.  
La documentation interactive (Swagger) est disponible sur `http://localhost:8000/docs`.

**Vérification rapide :**

```bash
curl http://localhost:8000/health
# {"status": "ok"}

curl -X POST http://localhost:8000/play \
  -H "Content-Type: application/json" \
  -d '{"observation": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]}'
# {"action": 0}
```

**Choix d'architecture** : FastAPI a été retenu pour sa validation automatique des données d'entrée via Pydantic (rejet immédiat d'une observation mal formée), sa documentation Swagger auto-générée, et ses performances asynchrones adaptées à des appels répétés depuis le GUI.

La logique RL est intégralement côté API : le GUI ne connaît pas le modèle, il envoie un état et reçoit une action. Cette séparation respecte le principe backend/frontend du brief.

### 2. Lancer le GUI

Dans un second terminal (l'API doit rester active) :

```bash
streamlit run gui/app.py
```

Le navigateur s'ouvre sur `http://localhost:8501`. Cliquer sur **"Lancer un épisode"** pour visualiser l'agent jouer en temps réel.

L'URL de l'API est configurable depuis la sidebar (défaut : `http://localhost:8000`).

**Choix technique** : Streamlit permet d'afficher les frames renvoyées par `env.render()` avec `st.image()` sans librairie supplémentaire, et d'appeler l'API via `requests` à chaque step. C'est suffisant pour la démonstration visuelle demandée par le brief.

### 3. Lancer le tableau de bord

Dans un troisième terminal :

```bash
streamlit run dashboard/app.py
```

Le tableau de bord s'ouvre sur `http://localhost:8502`. Depuis la sidebar :
- définir l'URL de l'API (défaut : `http://localhost:8000`)
- choisir le nombre d'épisodes à simuler (5 à 50)
- cliquer sur **"Lancer les épisodes"**

**Métriques affichées** : récompense moyenne ± écart-type, meilleure récompense, taux de succès (épisodes ≥ 200), courbe de récompense par épisode avec bande ± 1 σ, filtre dynamique par seuil de récompense, analyse de diversité des conditions initiales.

**Choix de design** : la bande ± 1 σ autour de la courbe permet de visualiser d'un coup d'œil la stabilité de la politique : une bande étroite indique un agent qui atterrit systématiquement. Le graphique de diversité des états initiaux permet de vérifier l'absence de surapprentissage : si l'agent performe bien sur des positions de départ variées, il a généralisé et non mémorisé.